# J-Space top-k tokens by layer

Reads saved capture files (no model load). For one example and token position, shows:

- the top-k J-Space vocabulary at each captured layer
- which layers each of those tokens appears in

Recording is chosen at **run time**, not in this notebook:

```bash
python -m gsm8k_jspace run --config configs/runs/small-smoke.yaml --capture
python -m gsm8k_jspace run --config configs/smoke.yaml --no-capture
```

YAML equivalent: `capture.enabled: true` or `false`, or overlays
`configs/experiments/capture-on.yaml` / `capture-off.yaml`.

Set `SAVE_OUTPUTS = True` below to write tables/figures into the run folder.
Use `SOURCE = "logit"` or `"model"` when the run stored those readouts (`full_sequence` capture).

In [ ]:
RUN_DIR = "latest_complete"
EXAMPLE_ID = None
POSITION = "last"
SOURCE = "jspace"
MAX_RANK = 10
SAVE_OUTPUTS = False

In [ ]:
from IPython.display import Markdown, display
from pathlib import Path
import pandas as pd

from gsm8k_jspace.analysis import (
    load_run,
    plot_token_layer_heatmap,
    token_layer_presence_table,
    token_layer_rank_grid,
    topk_by_layer_table,
)
from gsm8k_jspace.analysis.catalog import resolve_run

run_path = resolve_run(RUN_DIR)
run = load_run(run_path)
warnings = run.warn_incomplete()
example_id = EXAMPLE_ID or (
    run.completions[0]["example_id"] if run.completions else None
)
completion = next(
    (row for row in run.completions if row.get("example_id") == example_id),
    {},
)
recorded = run.has_captures()
save_outputs = bool(SAVE_OUTPUTS)

try:
    import ipywidgets as widgets
except Exception:
    widgets = None

example_widget = None
source_widget = None
position_widget = None
save_widget = None
if widgets is not None:
    example_options = [row["example_id"] for row in run.completions] or [example_id]
    example_widget = widgets.Dropdown(
        options=example_options,
        value=example_id if example_id in example_options else example_options[0],
        description="Example",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "90px"},
    )
    source_widget = widgets.Dropdown(
        options=["jspace", "logit", "model"],
        value=SOURCE,
        description="Source",
        style={"description_width": "90px"},
    )
    position_widget = widgets.Dropdown(
        options=["last", "all"],
        value=POSITION if POSITION in {"last", "all"} else "last",
        description="Position",
        style={"description_width": "90px"},
    )
    save_widget = widgets.Checkbox(
        value=save_outputs,
        description="Save tables/figures into this run folder",
        indent=False,
    )
    display(example_widget, source_widget, position_widget, save_widget)
    display(Markdown("_Change a control, then re-run the cells below._"))

meta = pd.DataFrame(
    [
        {
            "Run": run.manifest.get("run_id"),
            "Status": run.manifest.get("status"),
            "Example": example_id,
            "Position": POSITION,
            "Source": SOURCE,
            "Top-k": MAX_RANK,
            "Capture recorded": recorded,
            "capture.enabled": run.config.capture.enabled,
            "Capture file": completion.get("capture_file"),
            "Save outputs": save_outputs,
            "Warnings": "; ".join(warnings) if warnings else "none",
        }
    ]
)
display(Markdown(f"**Run** `{run_path}`"))
display(meta.style.hide(axis="index"))
if not recorded:
    display(
        Markdown(
            "**This run did not record J-Space captures.** "
            "GSM8K completions may still be here. To record top-k tokens by layer, re-run with "
            "`--capture` or `capture.enabled: true`."
        )
    )
if completion.get("generated_text"):
    display(Markdown("**Generated text**"))
    display(Markdown(f"```\n{completion['generated_text']}\n```"))


def _selected():
    chosen_example = example_widget.value if example_widget is not None else example_id
    chosen_source = source_widget.value if source_widget is not None else SOURCE
    chosen_position = position_widget.value if position_widget is not None else POSITION
    chosen_save = save_widget.value if save_widget is not None else save_outputs
    return chosen_example, chosen_source, chosen_position, chosen_save


def _output_dirs():
    tables = run.run_dir / "visualization" / "tables"
    figures = run.run_dir / "visualization" / "figures"
    tables.mkdir(parents=True, exist_ok=True)
    figures.mkdir(parents=True, exist_ok=True)
    return tables, figures

In [ ]:
example_id, source, position, save_outputs = _selected()
by_layer = pd.DataFrame(
    topk_by_layer_table(
        run,
        example_id=example_id,
        position=position,
        source=source,
        max_rank=MAX_RANK,
    )
)
if by_layer.empty:
    display(
        Markdown(
            "No top-k rows. This run may have used `--no-capture`, or it did not store "
            f"`top_{source}_tokens` at the selected position."
        )
    )
else:
    state = by_layer["state_token"].iloc[0]
    pos = by_layer["position"].iloc[0]
    display(
        Markdown(
            f"**Top-{MAX_RANK} {source} tokens at position `{pos}`** "
            f"(state token {state}, one row per layer)"
        )
    )
    display(by_layer.drop(columns=["position", "state_token"]).style.hide(axis="index"))
    if save_outputs:
        tables_dir, _ = _output_dirs()
        path = tables_dir / "topk_by_layer.csv"
        by_layer.to_csv(path, index=False)
        display(Markdown(f"Saved `{path}`"))

In [ ]:
example_id, source, position, save_outputs = _selected()
presence = pd.DataFrame(
    token_layer_presence_table(
        run,
        example_id=example_id,
        position=position,
        source=source,
        max_rank=MAX_RANK,
    )
)
if presence.empty:
    display(Markdown("No token-to-layer mapping for this selection."))
else:
    shown = presence.rename(
        columns={
            "token": "Token",
            "token_id": "Token id",
            "n_layers": "Layers",
            "layers": "Appears in layers",
            "best_rank": "Best rank",
            "mean_logit": "Mean logit",
            "n_hits": "Hits",
        }
    )
    if "Mean logit" in shown:
        shown["Mean logit"] = pd.to_numeric(shown["Mean logit"]).map(
            lambda value: f"{value:.2f}" if pd.notna(value) else ""
        )
    display(
        Markdown(
            "**Which layers each top-k token appears in** "
            "(sorted by best rank, then how many layers)"
        )
    )
    display(shown.style.hide(axis="index"))
    if save_outputs:
        tables_dir, _ = _output_dirs()
        path = tables_dir / "topk_token_layers.csv"
        presence.to_csv(path, index=False)
        display(Markdown(f"Saved `{path}`"))

In [ ]:
example_id, source, position, save_outputs = _selected()
grid = token_layer_rank_grid(
    run,
    example_id=example_id,
    position=position,
    source=source,
    max_rank=MAX_RANK,
)
grid_df = pd.DataFrame(grid)
if grid_df.empty:
    display(Markdown("No rank grid to plot."))
else:
    rank_cols = [col for col in grid_df.columns if col.startswith("L") and col[1:].isdigit()]
    display(Markdown("**Rank grid** (blank = token not in that layer's top-k)"))
    display(
        grid_df[["token", "best_rank", "n_layers", *rank_cols]]
        .rename(columns={"token": "Token", "best_rank": "Best rank", "n_layers": "Layers"})
        .style.hide(axis="index")
        .background_gradient(cmap="Blues_r", subset=rank_cols, vmin=1, vmax=MAX_RANK)
    )
    fig_path = None
    if save_outputs:
        _, figures_dir = _output_dirs()
        fig_path = figures_dir / "topk_rank_by_layer.png"
        tables_dir, _ = _output_dirs()
        grid_df.to_csv(tables_dir / "topk_rank_grid.csv", index=False)
    fig = plot_token_layer_heatmap(
        grid,
        title=f"{source} top-{MAX_RANK} rank by layer ({example_id})",
        max_tokens=40,
        path=fig_path,
    )
    display(fig)
    if fig_path is not None:
        display(Markdown(f"Saved `{fig_path}`"))